## Validation of model_predict_coupler_NCap_cap_matrix wtih Ansys Q3D

In [1]:
from squadds.simulations.objects import run_capn_LOM 
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal import designs, Dict, MetalGUI
import pandas as pd
import numpy as np

# Read in ML results and get design / simulation setup ready

In [2]:
## read in ML results
ML_results = pd.read_csv("predictions_and_errors_unscaled_one_hot.csv")

'''read in SQuADDS coupler-NCap-cap_matrix dataset'''
ncap_db = pd.read_json("coupler-NCap-cap_matrix.json")

''' design and simulation setup parameters from DB '''
sim_setup = ncap_db.sim_options.iloc[0]
SQuADDS_design_params = ncap_db.design.iloc[0]["design_options"] # substitute ML predicted design params into here 

In [3]:
results = pd.DataFrame({"Sample":[],
                   "ref_cap_matrix":[],
                   "pred_cap_matrix":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = np.unique(ML_results.sample_idx)

## Sweep through ML design results, simulate each design with Anysys Q3D, and save sim results
There are four multi-valued parameters in the SQuADDS coupler-NCap-cap_matrix dataset:
* design_options.cap_gap
* design_options.cap_width
* design_options.finger_length
* design_options.finger_count
  
The model only predicts these four design parameters. We substitute in the predicted parameters from the model direcly into Quantum (Qiskit) Metal and then simulate the design.

In [4]:
for sample in samples_to_test:

    ''' current testing sample '''
    this_device = ML_results[ML_results.sample_idx == sample]

    ''' extract predicted design parameters '''
    for param in np.unique(this_device.param):
        param_key = param.split(".")[1]
        design_value = this_device[this_device.param == param].pred_unscaled.iloc[0]
        
        if param_key == 'finger_count':
            SQuADDS_design_params[param_key] = np.round(design_value,0) 
        else:
            SQuADDS_design_params[param_key] = str(design_value*um)+"um"
        
    ''' create ML predicted design in Quantum (Qiskit) Metal '''
    design = designs.DesignPlanar()
    cplr = CapNInterdigitalTee(design,"NCap_{}".format(sample),options = SQuADDS_design_params)
    design.rebuild()

    ''' simulate predicted design '''
    pred_ansys_results = run_capn_LOM(design,cplr.options,sim_setup)

    pred_cap_matrix = pred_ansys_results[0]["sim_results"]

    # rename the keys to match the sim results keys to simplify analysis later
    ref_cap_matrix = {'C_top2top': this_device.top_to_top.iloc[0],
                      'C_top2bottom': this_device.top_to_bottom.iloc[0],
                      'C_top2ground': this_device.top_to_ground.iloc[0],
                      'C_bottom2bottom': this_device.bottom_to_bottom.iloc[0],
                      'C_bottom2ground': this_device.bottom_to_ground.iloc[0],
                      'C_ground2ground': this_device.ground_to_ground.iloc[0]}
    
    ''' save results '''
    results.loc[len(results)] = [sample,
                                 ref_cap_matrix,
                                 pred_cap_matrix]

the parameters ['run'] are unsupported, so they have been ignored


INFO 10:35AM [connect_project]: Connecting to Ansys Desktop API...
INFO 10:35AM [load_ansys_project]: 	Opened Ansys App
INFO 10:35AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 10:35AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project20
INFO 10:35AM [connect_design]: No active design found (or error getting active design).
INFO 10:35AM [connect]: 	 Connected to project "Project20". No design detected
INFO 10:35AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d [Solution type: Q3D]
WARNING 10:35AM [connect_setup]: 	No design setup detected.
WARNING 10:35AM [connect_setup]: 	Creating Q3D default setup.
INFO 10:35AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:35AM [get_setup]: 	Opened setup `lom_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:35AM [analyze]: Analyzing setup lom_setup
INFO 10:36AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppDa

the parameters ['run'] are unsupported, so they have been ignored


INFO 10:36AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d1 [Solution type: Q3D]
WARNING 10:36AM [connect_setup]: 	No design setup detected.
WARNING 10:36AM [connect_setup]: 	Creating Q3D default setup.
INFO 10:36AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:36AM [get_setup]: 	Opened setup `lom_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:36AM [analyze]: Analyzing setup lom_setup
INFO 10:37AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\10\tmpdvln3409.txt, C, , lom_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR\ansys.py: 1498
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR\ansys.py: 1505
INFO 10:37AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\10\tmp4al40rdb.txt, C, , lom_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 500000

the parameters ['run'] are unsupported, so they have been ignored


...

INFO 11:24AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d46 [Solution type: Q3D]
INFO 11:24AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:24AM [connect]: 	Connected to project "Project20" and design "LOMv2.01_q3d46" 😀 

INFO 11:24AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d47 [Solution type: Q3D]
WARNING 11:24AM [connect_setup]: 	No design setup detected.
WARNING 11:24AM [connect_setup]: 	Creating Q3D default setup.
INFO 11:24AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:24AM [get_setup]: 	Opened setup `lom_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:24AM [analyze]: Analyzing setup lom_setup
INFO 11:25AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\10\tmpzotikwby.txt, C, , lom_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR

the parameters ['run'] are unsupported, so they have been ignored


INFO 11:25AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d47 [Solution type: Q3D]
INFO 11:25AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:25AM [connect]: 	Connected to project "Project20" and design "LOMv2.01_q3d47" 😀 

INFO 11:25AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d48 [Solution type: Q3D]
WARNING 11:25AM [connect_setup]: 	No design setup detected.
WARNING 11:25AM [connect_setup]: 	Creating Q3D default setup.
INFO 11:25AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:25AM [get_setup]: 	Opened setup `lom_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:25AM [analyze]: Analyzing setup lom_setup
INFO 11:26AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\10\tmpek5_ozrf.txt, C, , lom_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR

the parameters ['run'] are unsupported, so they have been ignored


INFO 11:26AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d48 [Solution type: Q3D]
INFO 11:26AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:26AM [connect]: 	Connected to project "Project20" and design "LOMv2.01_q3d48" 😀 

INFO 11:26AM [connect_design]: 	Opened active design
	Design:    LOMv2.01_q3d49 [Solution type: Q3D]
WARNING 11:26AM [connect_setup]: 	No design setup detected.
WARNING 11:26AM [connect_setup]: 	Creating Q3D default setup.
INFO 11:26AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:26AM [get_setup]: 	Opened setup `lom_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:26AM [analyze]: Analyzing setup lom_setup
INFO 11:27AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\10\tmph6l3_pkk.txt, C, , lom_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
 C:\Users\firas\miniconda3\envs\squadds041\Lib\site-packages\pyEPR

# Save the data

In [7]:
results.to_csv("NCap_simulation_results.csv",index=False) # save data for later analysis with validation_01_data_analysis.ipynb